# 11. GARCH model — one year (Method #1)
The second benchmark, **GARCH(1,1)**, is the industry-standard volatility model and is famously hard to beat. It needs the library `arch`. As in notebook 09, we build it for one year: estimate on data before 2007, test on 2007.

* **Needs:** `work/dataset.csv`. **Makes:** nothing new.

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

The library `arch` is not in Colab by default, so we install it (like `pip install category_encoders` in the Life Expectancy notebook):

In [ ]:
!pip install -q arch

In [3]:
from pandas import read_csv
dataset = read_csv(folder + 'work/dataset.csv', index_col=0, parse_dates=True)
dataset = dataset.loc[:'2023-08-31']        # main sample
print(dataset.shape)

(5243, 17)


## The idea
$$\sigma^2_{tomorrow} = \omega + \alpha \cdot \text{shock}_{today}^2 + \beta \cdot \sigma^2_{today}$$
Tomorrow's variance = a small constant + a reaction to today's surprise + most of today's variance carried forward.

A calculation by hand (returns in %): $\omega$ = 0.02, $\alpha$ = 0.09, $\beta$ = 0.90; today's variance is 1, and today the market fell 3%.

In [4]:
omega = 0.02
alpha = 0.09
beta = 0.90
variance_today = 1.0
shock_today = -3.0
variance_tomorrow = omega + alpha * shock_today ** 2 + beta * variance_today
print('variance tomorrow =', variance_tomorrow)

variance tomorrow = 1.73


A 3% fall pushes tomorrow's variance from 1 to 1.73. $\alpha + \beta$ close to 1 means the shock fades only slowly.

## The data: daily returns in per cent
GARCH works on the daily returns (×100, in per cent, which helps the estimation).

In [5]:
returns = dataset['ret'].dropna() * 100
returns.head()

2003-01-02    2.823328
2003-01-03    1.366236
2003-01-06    0.550575
2003-01-07   -0.137115
2003-01-08   -0.607838
Name: ret, dtype: float64

# GARCH(1,1)
## Modelling
Student-t errors (`dist='t'`) allow for crashes (fat tails). `last_obs='2007-01-01'` means: estimate using data **before** 2007 only.

In [6]:
from arch import arch_model
GARCH_model1 = arch_model(returns, mean='Constant', vol='GARCH', p=1, q=1, dist='t')   # building
GARCH_fit1 = GARCH_model1.fit(last_obs='2007-01-01', disp='off')                        # training
print(GARCH_fit1.summary())

                        Constant Mean - GARCH Model Results                         
Dep. Variable:                          ret   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -1154.12
Distribution:      Standardized Student's t   AIC:                           2318.24
Method:                  Maximum Likelihood   BIC:                           2342.85
                                              No. Observations:                 1013
Date:                      Thu, Oct 01 2026   Df Residuals:                     1012
Time:                              04:31:37   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu        

In [7]:
print('alpha (reaction) =', round(GARCH_fit1.params['alpha[1]'], 3))
print('beta (memory)    =', round(GARCH_fit1.params['beta[1]'], 3))
print('alpha + beta     =', round(GARCH_fit1.params['alpha[1]'] + GARCH_fit1.params['beta[1]'], 3))
print('nu (tails)       =', round(GARCH_fit1.params['nu'], 1))

alpha (reaction) = 0.06
beta (memory)    = 0.897
alpha + beta     = 0.957
nu (tails)       = 6.2


## Testing
From every day of 2007 the model forecasts the variance 1, 2, 3, 4 and 5 days ahead (`horizon=5`). The parameters stay fixed, but each day's forecast uses that day's newest return.

In [8]:
test_days = dataset.loc['2007-01-01':'2007-12-31'].dropna(subset=['rv5_fwd']).index
forecast1 = GARCH_fit1.forecast(horizon=5, start=test_days[0], reindex=False)     # testing
forecast1.variance.head()

,h.1,h.2,h.3,h.4,h.5
2007-01-02,0.581251,0.582489,0.583674,0.584808,0.585893
2007-01-03,0.649109,0.647415,0.645794,0.644243,0.642759
2007-01-04,0.620858,0.620385,0.619932,0.619498,0.619084
2007-01-05,0.587928,0.588878,0.589787,0.590657,0.591489
2007-01-08,0.630085,0.629213,0.628379,0.627580,0.626817


Columns `h.1` … `h.5` are the variances 1 to 5 days ahead. Next week's variance is their **sum**. Because the returns were in per cent, we divide by 100² = 10,000 to come back to decimals.

In [9]:
y_pred1 = forecast1.variance.loc[test_days].sum(axis=1) / 10000
y_pred1.head()

2007-01-02    0.000292
2007-01-03    0.000323
2007-01-04    0.000310
2007-01-05    0.000295
2007-01-08    0.000314
dtype: float64

## Evaluation

In [10]:
from numpy import log
y_test = dataset.loc[test_days, 'rv5_fwd']
ratio = y_test / y_pred1
QLIKE1 = (ratio - log(ratio) - 1).mean()
print('QLIKE (GARCH, 2007) =', round(QLIKE1, 3))

QLIKE (GARCH, 2007) = 0.518


In [11]:
saved = read_csv(folder + 'results/benchmark_forecasts.csv', index_col=0, parse_dates=True)
print('largest difference from the saved GARCH forecasts:', (y_pred1 - saved.loc[test_days, 'GARCH']).abs().max())

largest difference from the saved GARCH forecasts: 9.947554932554503e-17


In 2007 GARCH (0.52) is more accurate than HAR (0.66 in notebook 09): it reacts faster when the crisis starts in August.